# What Preprocessing Leakage Cost a Startup Success Model

*8,837 Crunchbase companies. Preprocessing fit inside each fold scores 0.794265, fit on all rows first, 0.794240.*

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

dataset = "/kaggle/input/big-startup-secsees-fail-dataset-from-crunchbase/big_startup_secsees_dataset.csv"
if not os.path.exists(dataset):
    dataset = "big_startup_secsees_dataset.csv"
df_original = pd.read_csv(dataset)

In [ ]:
df_original.head()

Every step below runs on the full dataset, before any train/test split. That is only safe under one rule: a step may run before the split only if it is row-local, meaning the value it produces for a row can be worked out from that row alone. A step that has to read other rows to produce its answer belongs inside the pipeline, where it is refit on the training rows of each fold and never sees held-out data.

The two operations on `funding_total_usd` sit either side of that line. Flagging a row as missing reads one cell of one row, so it returns the same answer on the full dataset as it would on a single row, and `funding_missing` is built here. Filling that missing value with the median reads every row to produce a single number. Computed above the split, that median is measured from the training rows and the held-out rows together, and the test set has quietly shaped the data the model trains on. This is preprocessing leakage: not the model reading a test row, but a value measured from test rows being stitched into the training data.

Dropping companies with no resolved outcome, mapping `status` to a 0/1 target, parsing `-` to `NaN`, subtracting dates, and taking the first entry out of `category_list` are all row-local, so they stay here. Anything that learns a quantity from the data, the median, the scaler's mean and spread, the encoder's category ranking, is held back for the pipeline.

In [ ]:
df = df_original[~(df_original['status'] == 'operating')]

In [ ]:
df.shape

In [ ]:
df['success'] = df['status'].map({
    'ipo': 1,
    'acquired': 1,
    'closed': 0
})

In [ ]:
df = df.drop(columns=['permalink', 'name', 'homepage_url', 'status', 'state_code', 'region', 'city'])

In [ ]:
df.isna().sum()

In [ ]:
df = df.dropna(subset=['founded_at', 'first_funding_at'])

In [ ]:
cols = ['category_list', 'country_code']

for c in cols:
    print(df[c].value_counts(dropna=False).head(10))

In [ ]:
df.dtypes

In [ ]:
mask = pd.to_numeric(df['funding_total_usd'], errors='coerce').isna()
df.loc[mask, 'funding_total_usd'].value_counts()

In [ ]:
df['funding_total_usd'] = pd.to_numeric(df['funding_total_usd'], errors='coerce')

In [ ]:
df['funding_total_usd'].value_counts(dropna=False)

In [ ]:
df['funding_missing'] = df['funding_total_usd'].isna()

In [ ]:
dates = ['founded_at', 'first_funding_at', 'last_funding_at']
df[dates] = df[dates].apply(pd.to_datetime, errors='coerce')

In [ ]:
df.dtypes

In [ ]:
df['days_to_first_funding'] = (df['first_funding_at'] - df['founded_at']).dt.days
df['funding_duration'] = (df['last_funding_at'] - df['first_funding_at']).dt.days

In [ ]:
df = df.drop(columns=['founded_at', 'first_funding_at', 'last_funding_at'])

Both new columns come from subtraction, which can produce values the calendar cannot. Nulls, negatives and zeros each need checking, and a negative and a zero turn out to mean very different things.

In [ ]:
days = ['days_to_first_funding', 'funding_duration']

summary = pd.DataFrame({
    'null': df[days].isna().sum(),
    'negative': (df[days] < 0).sum(),
    'zero': (df[days] == 0).sum()
})

display(summary)
print(f"Companies with a single funding round: {(df['funding_rounds'] == 1).sum()}")

A negative `days_to_first_funding` records funding arriving before the company existed, which cannot happen, so those 763 rows are broken records and come out. A zero `funding_duration` is a different kind of finding. It is not an impossible value: it describes a company whose first and last funding fell on the same day. Dropping those would cut more than half the remaining rows and, worse, would remove a real category of company rather than a mistake, so they stay.

5,311 companies show a `funding_duration` of zero, but only 5,280 raised a single round. A zero duration normally means one round and nothing after, so those two counts should describe the same companies, and 31 of them do not. That leaves one explanation worth testing: a company that raised more than one round on the same day, and never raised again.

In [ ]:
mask = (df['funding_duration'] == 0) & (df['funding_rounds'] > 1)
comparison = df.loc[mask, ['funding_duration', 'funding_rounds']]
display(comparison)
print(f'Companies with funding duration "zero" but more than one funding round: {mask.sum()}')

All 31 raised every round they ever raised on the same day. Together with the 5,280 single-round companies, they account for all 5,311 zero-duration rows, so the gap is fully explained and no third kind of company is left to look for.

In [ ]:
df = df[df['days_to_first_funding'] >= 0]

In [ ]:
# class proportions
(df['success'].value_counts(normalize=True)*100).round(1)

In [ ]:
print(f"Success rate of companies where `funding_duration` is zero: {df.loc[df['funding_duration'] == 0, 'success'].mean():.2f}")

Among the companies where `funding_duration` is zero, 54% failed. Among all companies in the dataset, 41.1% failed. Zero-duration companies fail more often than companies in general, so the model learns to read a `funding_duration` of zero as a sign of failure.

The model can only read a zero that way because every company in this data has already finished. `funding_duration` counts the days between a company's first and last funding round, and the last round is only the last one because the company shut down or was acquired and never raised again. Companies still operating had to be dropped, since a company with no outcome has no label to train on. Dropping them leaves a column where every duration is complete: no missing values, no negatives, nothing out of range. Inspecting the column finds nothing wrong.

The problem appears when the model is used on a company that is still running. That company's last funding round is only the last one so far. A startup that raised its first round two months ago has a `funding_duration` of zero, the same value carried by the finished companies that failed 54% of the time in training. For the startup, a zero means early, not finished.

Cross-validation does not catch this, and it cannot. Every fold is drawn from the same pool of finished companies, so a zero carries the same wrong meaning in the training folds and in the validation folds. The score stays high, and nothing in it points at the column.

In [ ]:
df['category'] = df['category_list'].str.split('|').str[0]
df = df.drop(columns='category_list')

In [ ]:
print(f"Category (Null): {df['category'].isna().sum()}")

In [ ]:
X = df.drop(columns='success')
y = df['success']

`stratify=y` splits the data so that both halves carry the same class balance as the full dataset, 58.9% success to 41.1% failure. Without it the split is drawn at random, and the class balance in each half can drift from the dataset's, leaving a test set that measures a slightly different population than the one the model trained on.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

In [ ]:
print("funding_total_usd")
print(f"{'Mean:':<7}{df['funding_total_usd'].mean():>10,.0f}")
print(f"{'Median:':<7}{df['funding_total_usd'].median():>10,.0f}")

# Here

`funding_total_usd` has NaN rows and needs to be filled with either mean or median. Median makes more sense here since mean is 6 times larger than median, aka heavily right skewed, and it will inflate the rows with missing values. To avoid data leakage, median filling process is put into hte pipeline where it is computed only from train data.

For one hot encoding, maximum categories is set at 11 the most frequent 10 plus infrequent column if a category that wasn't in the training data appears.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(max_categories=11, handle_unknown='infrequent_if_exist'))
])

In [ ]:
ct = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, ['funding_total_usd', 'funding_rounds', 'days_to_first_funding', 'funding_duration', 'funding_missing']),
        ('cat', categorical_transformer, ['country_code', 'category'])
    ])

In [ ]:
pipeline = Pipeline(steps=[
    ('prep', ct),
    ('model', LogisticRegression())
])

In [ ]:
score = cross_val_score(
    pipeline,
    X_train, y_train,
    cv=5,
    scoring='roc_auc'
)
print(score)
print(f"Mean 5-fold CV ROC-AUC: {score.mean():.6f}")

## Measuring the leak

In [ ]:
X_leaky = ct.fit_transform(X)

In [ ]:
X_leaky_train, X_leaky_test, y_leaky_train, y_leaky_test = train_test_split(
    X_leaky, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
leaky_score = cross_val_score(
    LogisticRegression(),
    X_leaky_train, y_leaky_train,
    cv=5,
    scoring='roc_auc'
)
print(leaky_score)
print(f"Mean 5-fold CV ROC-AUC: {leaky_score.mean():.6f}")

In [ ]:
print(leaky_score - score)

In conclusion, there is barely any differnce on the AUC score leaking the data vs. using a pipeline. 